## STEP 1 - SLM AUGMENTATION
#### In this step, we're going to generate augmented samples using SLMs to enhance the vocabulary representativeness of each sample, based on previous data analysis : ) 

In [ ]:
import pandas as pd
from pipeline.slm_augmenter import SLMAugmenter
from pipeline.preprocessor import Preprocessor

In [17]:
raw_dataset = pd.read_csv('data/raw_datasets/mbti_1.csv')

Before anything, let's clean some HTML tags from the texts

In [18]:
pre = Preprocessor('posts')

In [19]:
no_html_dataset = pre.remove_html(raw_dataset)

In [20]:
no_html_dataset.head()

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...


Now, let's run it through the SLM of choice (just change the model_name)

In [21]:
augmenter = SLMAugmenter(texts_column='posts', label_column='type', model_name='qwen2.5:1.5b')

Verificando/Baixando o modelo 'qwen2.5:1.5b'... Isso pode demorar na 1ª vez se você ainda não baixou.


In [ ]:
df_final = augmenter.augment_minority_classes(
    df=no_html_dataset, 
    checkpoint_file='data/checkpoints/augmentation_checkpoint_qwen.csv',
    sample_size=5 #This is just an example. Remove the sample_size parameter for full execution.
)


Iniciando augmentation local com qwen2.5:1.5b...
Carregando checkpoint salvo em 'data/checkpoints/augmentation_checkpoint_qwen.csv'...
5 textos já processados encontrados. Retomando de onde parou...
Augmentation finalizado com sucesso!


In [23]:
df_final.head()

,type,posts,augmented_posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,<NA>
1,ENTP,'I'm finding the lack of me in these posts ver...,I am finding a noticeable absence of myself in...
2,INTP,'Good one _____ https://www.youtube.com/wat...,<NA>
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...",<NA>
4,ENTJ,'You're fired.|||That's another silly misconce...,"""You're fired."" ||| That's another common misc..."


Now, we gotta apply the textual preprocessing


In [25]:
df_final['augmented_posts'] = df_final['augmented_posts'].fillna(df_final['posts'])

In [26]:
df_final.head()

,type,posts,augmented_posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...,I am finding a noticeable absence of myself in...
2,INTP,'Good one _____ https://www.youtube.com/wat...,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...","'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...,"""You're fired."" ||| That's another common misc..."


In [ ]:
pre_augmented = Preprocessor(texts_column= 'augmented_posts')

In [28]:
pre_augmented.preprocess_complete(df_final).head()

,type,posts,augmented_posts,augmented_posts_clean
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,moment sportscenter top ten play prankswhat ha...
1,ENTP,'I'm finding the lack of me in these posts ver...,I am finding a noticeable absence of myself in...,i finding noticeable absence post sex monotono...
2,INTP,'Good one _____ https://www.youtube.com/wat...,'Good one _____ https://www.youtube.com/wat...,good one course i say i know thats blessing cu...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...","'Dear INTP, I enjoyed our conversation the o...",dear i enjoyed conversation day esoteric gabbi...
4,ENTJ,'You're fired.|||That's another silly misconce...,"""You're fired."" ||| That's another common misc...",youre fired thats another common misconception...


In [ ]:
pre_augmented.to_csv("data/augmented_datasets/qwen_augmented_dataset.csv")